# Agent Notebook: ASP Causal Study
## Gemini API Agent Evaluation

This notebook calls the Gemini API with the analysis prompt, executes the
returned code, saves the results to `agent_results.pkl`, and lets you validate
the results using `unit_test_notebook.ipynb`.


In [ ]:
!pip install -r requirements.txt -q

import os, re, warnings, textwrap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from linearmodels.panel import PanelOLS
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
import google.generativeai as genai
warnings.filterwarnings('ignore')

df = pd.read_csv('asp_study_data.csv')
print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("\nFirst 3 rows:")
print(df.head(3).to_string())
print("\nColumn dtypes:")
print(df.dtypes.to_string())
print("\nBasic stats:")
print(df[['age','severity_score','antibiotic_duration',
          'time_to_improvement','event_observed','asp_active','mortality']].describe().round(2).to_string())


## Prompt Sent to Gemini

You are an expert data scientist specialising in clinical research and causal inference.

**CONTEXT:**
A consortium of 15 hospitals across five regions participated in a longitudinal observational
study (2019-2022) to evaluate the real-world effectiveness of Antimicrobial Stewardship
Programs (ASPs). ASPs were voluntarily adopted by hospitals at different points: some in 2020,
others in 2021, and some never adopted. The dataset `asp_study_data.csv` (already loaded as
the pandas DataFrame `df`) contains one row per patient admission.

**KEY COLUMNS:**
- `patient_id`, `hospital_id`, `year`, `asp_active` (1 = ASP in effect at time of admission)
- `asp_adoption_year` (2020, 2021, or 9999 for never-adopted)
- `time_to_improvement` (days until clinical improvement, the survival time)
- `event_observed` (1 = patient reached clinical improvement; 0 = right-censored at study end)
- `antibiotic_duration` (days of antibiotic use, primary policy outcome)
- `mortality`, `readmission_30d`, `c_diff_infection` (secondary outcomes)
- `age`, `age_group`, `sex`, `severity_score`, `comorbidity_score`, `immune_compromised`
- `infection_type`, `ward`, `region`, `hospital_size`

**TASK:**
Produce a rigorous causal analysis of ASP effectiveness. Your analysis must address all four objectives:

1. **Survival Analysis:** Quantify how ASP adoption affects time-to-clinical-improvement,
   accounting for patients who have not yet reached the endpoint by the study end.
   Verify that your survival model respects the censoring structure of the data.

2. **Causal Impact (DiD):** Estimate the causal impact of ASP on antibiotic duration.
   Account for the voluntary and staggered nature of hospital adoption; validate the
   causal identification assumptions before reporting your treatment effect estimate.

3. **Treatment Heterogeneity:** Identify which patient subgroups derive the greatest
   benefit from ASP. Quantify individual-level treatment effects using a principled
   estimation framework.

4. **Hierarchical Structure:** Assess whether unmeasured hospital-level factors
   systematically confound the analysis. Produce model estimates that account for the
   nested structure of patients within hospitals, and report the degree of within-hospital
   clustering.

**REQUIRED OUTPUT VARIABLES** (must be defined for the unit tests):
`cox_hr_asp` (float), `ph_test_performed` (bool), `did_panel` (DataFrame, 60 rows),
`pre_trend_coeff` (float), `pre_trend_pvalue` (float), `did_att_estimate` (float),
`staggered_did_used` (bool), `cate_estimates` (numpy array, one value per patient),
`meta_learner_used` (str: 'T-learner', 'X-learner', or 'both'),
`icc_value` (float), `mixed_model_aic` (float), `naive_ols_aic` (float),
`mixed_asp_coeff` (float), `random_effect_variance` (float),
`cluster_robust_se_used` (bool)


In [ ]:
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not set.\n"
        "  Local : add GEMINI_API_KEY=your_key to a .env file\n"
        "  Colab : Secrets panel (key icon) -> name: GEMINI_API_KEY"
    )

genai.configure(api_key=GEMINI_API_KEY)

data_context = f"""
Dataset shape    : {df.shape[0]:,} rows x {df.shape[1]} columns
Columns          : {list(df.columns)}
Dtypes           :
{df.dtypes.to_string()}

Sample (first 5 rows):
{df.head(5).to_string()}

Key statistics:
{df[['age','severity_score','antibiotic_duration','time_to_improvement',
     'event_observed','asp_active','mortality']].describe().round(2).to_string()}

Unique hospitals : {df['hospital_id'].nunique()}
ASP adoption years (unique): {sorted(df['asp_adoption_year'].unique())}
Years in study   : {sorted(df['year'].unique())}
Censoring rate   : {1 - df['event_observed'].mean():.1%}
"""

FULL_PROMPT = """
You are an expert data scientist specialising in clinical research and causal inference.

CONTEXT:
A consortium of 15 hospitals across five regions participated in a longitudinal observational
study (2019-2022) to evaluate the real-world effectiveness of Antimicrobial Stewardship
Programs (ASPs). ASPs were voluntarily adopted at different points (2020, 2021, or never).
The DataFrame `df` is already loaded and contains one row per patient admission.

""" + data_context + """

TASK:
Produce a rigorous causal analysis of ASP effectiveness. Address all four objectives:

1. SURVIVAL ANALYSIS: Quantify how ASP affects time-to-clinical-improvement, accounting
   for patients who have not yet reached the endpoint (right-censored observations).
   Use the Kaplan-Meier estimator and Cox Proportional Hazards model. Test the
   proportional hazards assumption.

2. CAUSAL IMPACT (DiD): Estimate the causal impact of ASP on antibiotic_duration.
   Account for the voluntary, staggered nature of adoption. Build a hospital-year panel,
   test parallel pre-trends, run TWFE regression, and produce an event study.

3. TREATMENT HETEROGENEITY: Identify which patients benefit most. Estimate individual-level
   CATE using a T-Learner or X-Learner. Use gradient boosting or random forests as base learners.

4. HIERARCHICAL MODELING: Fit a mixed-effects linear model with hospital as a random
   intercept. Calculate and report the ICC. Compare AIC against naive OLS.

CONSTRAINTS:
- event_observed=0 means censored. Do NOT treat these as failures.
- Validate parallel trends before reporting DiD estimates.
- ICC = random_effect_variance / (random_effect_variance + residual_variance).

REQUIRED OUTPUT VARIABLES (define these exactly with these names):
- cox_hr_asp : float
- ph_test_performed : bool = True
- did_panel : DataFrame with exactly 60 rows
- pre_trend_coeff : float
- pre_trend_pvalue : float
- did_att_estimate : float (should be negative)
- staggered_did_used : bool = True
- cate_estimates : numpy array, one value per patient
- meta_learner_used : str, one of 'T-learner', 'X-learner', or 'both'
- icc_value : float
- mixed_model_aic : float
- naive_ols_aic : float
- mixed_asp_coeff : float
- random_effect_variance : float
- cluster_robust_se_used : bool = True

Return ONLY valid, executable Python code. No markdown, no explanations, no code fences.
The code runs via exec() where `df` and all imported libraries are already available.
"""

MODELS_TO_TRY = [
    'gemini-2.5-pro',
    'gemini-2.5-flash',
    'gemini-1.5-pro',
    'gemini-1.5-flash',
]
model = None
selected_model_name = None
for _m in MODELS_TO_TRY:
    try:
        _test = genai.GenerativeModel(_m)
        _test.generate_content("hi", generation_config=genai.types.GenerationConfig(max_output_tokens=5))
        model = genai.GenerativeModel(
            model_name=_m,
            generation_config=genai.types.GenerationConfig(
                temperature=0.1,
                max_output_tokens=65536,
            )
        )
        selected_model_name = _m
        break
    except Exception as _e:
        print(f"  {_m} unavailable: {type(_e).__name__}")
if model is None:
    raise RuntimeError("No Gemini model available. Check your API key and billing.")
print("Model selected : " + selected_model_name)
print("Prompt length  : " + str(len(FULL_PROMPT)) + " characters")
print()

def _extract_code(text):
    text = text.strip()
    if '```python' in text:
        m = re.search(r'```python\n?(.*?)```', text, re.DOTALL)
        return m.group(1).strip() if m else text
    if '```' in text:
        m = re.search(r'```\n?(.*?)```', text, re.DOTALL)
        return m.group(1).strip() if m else text
    return text

MAX_ATTEMPTS = 5
chat = model.start_chat(history=[])
final_code = None

print("=" * 70)
print(f"AGENT EXECUTION: {selected_model_name}")
print("=" * 70)

response = chat.send_message(FULL_PROMPT)

for _attempt in range(1, MAX_ATTEMPTS + 1):
    print(f"\nAttempt {_attempt} / {MAX_ATTEMPTS}")
    print("-" * 40)

    _code = _extract_code(response.text)
    print(_code)

    try:
        exec(compile(_code, f'<gemini_attempt_{_attempt}>', 'exec'), globals())
        print(f"\nCode executed successfully on attempt {_attempt}")
        final_code = _code
        break
    except Exception as _exc:
        _err = f"{type(_exc).__name__}: {_exc}"
        print(f"\nError on attempt {_attempt}: {_err}")

        if _attempt < MAX_ATTEMPTS:
            print("   Sending error back to Gemini for correction...")
            _feedback = f"""Your code raised this error when executed:

{_err}

AVAILABLE PACKAGES (do not use anything else):
lifelines, linearmodels, statsmodels, sklearn (scikit-learn),
econml, pandas, numpy, scipy, matplotlib, seaborn, re, os

Do NOT use !pip install. All packages are already installed.
Do NOT use: causalml, pyfixest, or any other unlisted package.

Provide the complete corrected Python code only. No explanations."""
            response = chat.send_message(_feedback)
        else:
            print(f"\nAgent failed to produce working code after {MAX_ATTEMPTS} attempts.")
            print("Proceeding to save partial results...")


## Save Agent Results

In [ ]:
import pickle
import numpy as np

def _safe_get(name, default=None):
    return globals().get(name, default)

agent_results = {
    'df':                     _safe_get('df'),
    'cox_hr_asp':             _safe_get('cox_hr_asp'),
    'ph_test_performed':      _safe_get('ph_test_performed'),
    'did_panel':              _safe_get('did_panel'),
    'pre_trend_coeff':        _safe_get('pre_trend_coeff'),
    'pre_trend_pvalue':       _safe_get('pre_trend_pvalue'),
    'did_att_estimate':       _safe_get('did_att_estimate'),
    'staggered_did_used':     _safe_get('staggered_did_used'),
    'cate_estimates':         np.array(_safe_get('cate_estimates', [])),
    'meta_learner_used':      _safe_get('meta_learner_used'),
    'icc_value':              _safe_get('icc_value'),
    'mixed_model_aic':        _safe_get('mixed_model_aic'),
    'naive_ols_aic':          _safe_get('naive_ols_aic'),
    'mixed_asp_coeff':        _safe_get('mixed_asp_coeff'),
    'random_effect_variance': _safe_get('random_effect_variance'),
    'cluster_robust_se_used': _safe_get('cluster_robust_se_used'),
}

with open('agent_results.pkl', 'wb') as f:
    pickle.dump(agent_results, f)

defined = [k for k, v in agent_results.items() if v is not None]
missing = [k for k, v in agent_results.items() if v is None]
print("Saved agent_results.pkl")
print(f"Defined : {defined}")
if missing:
    print(f"Missing : {missing}")
